In [ ]:
import arcpy
from arcpy.sa import *

# 1. 設置環境與權限
arcpy.CheckOutExtension("Spatial")
# 允許覆寫已存在的檔案，避免因為檔名重複而報錯卡死
arcpy.env.overwriteOutput = True 

# 2. 獲取當前專案
aprx = arcpy.mp.ArcGISProject("CURRENT")
arcpy.env.workspace = aprx.defaultGeodatabase
print(f"工作空間：{arcpy.env.workspace}")

# 設置並行處理（建議先設為 0 或 50% 觀察是否穩定）
# 有時候 100% 並行反而會因為 IO 寫入衝突導致最後合併時卡住
arcpy.env.parallelProcessingFactor = "50%"

input_raster = "dome_crveice_test.tif"
output_name = r"D:\ArcGIS\Morpho_Spatial_Demography\Focal_dome_05.tif"

# 3. 定義鄰域
# 請確認：如果你的圖層單位是公尺，0.05 代表 5 公分。
# 如果你的 Cell Size (像素大小) 大於 0.05，這個運算將失去意義。
neighborhood = NbrRectangle(0.05, 0.05, "MAP")

print(f"正在執行 Median 運算... 請稍候 (資料量大時需數分鐘至數十分鐘)")

try:
    # 停用自動建立金字塔與統計資料
    arcpy.env.pyramid = "NONE"
    arcpy.env.rasterStatistics = "NONE"
    # 執行工具
    out_focal = FocalStatistics(input_raster, neighborhood, "MEDIAN", "DATA")
    
    # 儲存
    print("運算完成，正在存檔至 GDB...")
    out_focal.save(output_name)
    print(f"✅ 成功！結果檔案：{output_name}")

except arcpy.ExecuteError:
    # 捕捉 ArcGIS 特有的錯誤訊息
    print(arcpy.GetMessages(2))
except Exception as e:
    print(f"❌ 發生錯誤: {e}")

finally:
    arcpy.CheckInExtension("Spatial")

工作空間：D:\ArcGIS\Morpho_Spatial_Demography\MSD_Hawaii.gdb
正在執行 Median 運算... 請稍候 (資料量大時需數分鐘至數十分鐘)
運算完成，正在存檔至 GDB...
